# LC 39 — Combination Sum
**Difficulty:** Medium | **Pattern:** Backtracking

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Sort candidates once, then explore a
decision tree where each node either <em>includes</em> the current
candidate (and stays at the same index to allow reuse) or
<em>skips</em> to the next index. Pruning with <code>break</code>
when <code>candidate &gt; remaining</code> keeps the search tight.
</div>

## Official Problem Statement

Given an array of **distinct** integers `candidates` and a target
integer `target`, return a list of all **unique combinations** of
`candidates` where the chosen numbers sum to `target`.

You may return the combinations in **any order**. The **same** number
may be chosen from `candidates` an **unlimited number of times**.
Two combinations are unique if the frequency of at least one of the
chosen numbers is different.

**Constraints:**
- `1 <= candidates.length <= 30`
- `2 <= candidates[i] <= 40`
- All elements of `candidates` are **distinct**.
- `1 <= target <= 40`

## What This Is Actually Asking

Find every multiset of numbers from `candidates` that adds up to
`target`. A number can appear more than once in a combination.
Order within a combination does not matter — `[2,2,3]` and
`[3,2,2]` are the same combination.

The key challenge: avoid duplicate combinations while still allowing
the same candidate to be reused multiple times.

## Walk Through an Example by Hand

`candidates = [2, 3, 6, 7]`, `target = 7`

After sorting: `[2, 3, 6, 7]`

Start with `path=[]`, `remain=7`, `start=0`:

1. Pick `2` → `path=[2]`, `remain=5`, recurse from index 0
   - Pick `2` → `path=[2,2]`, `remain=3`, recurse from 0
     - Pick `2` → `path=[2,2,2]`, `remain=1`, recurse from 0
       - `2 > 1` → break (prune)
     - Pick `3` → `path=[2,2,3]`, `remain=0` ✓ **add**
     - `6 > 3` → break
   - Pick `3` → `path=[2,3]`, `remain=2`, recurse from 1
     - `3 > 2` → break
   - `6 > 5` → break
2. Pick `3` → `path=[3]`, `remain=4`, recurse from 1
   - Pick `3` → `path=[3,3]`, `remain=1` → `3>1` break
   - `6 > 4` → break
3. Skip `6` (6 < 7, recurse) → no valid paths before 7
4. Pick `7` → `remain=0` ✓ **add**

Result: `[[2,2,3], [7]]`

## The Picture

Decision tree for `candidates=[2,3,6,7]`, `target=7`:

```
                    []
          /        /       \      \
        [2]       [3]      [6]    [7]*
       /   \      /  \
    [2,2] [2,3] [3,3] [3,6>7?]
    /   \    |
[2,2,2][2,2,3]* [2,3] remain=2
  |              |
remain=1       remain=2
 prune          3>2 prune

* = valid combination found (remain == 0)
```

**Key rules:**
- Each level picks from index `i` onward (no left duplicates)
- Same index `i` is passed on recursive call → allows reuse
- `break` prunes the branch when `cand > remain` (sorted list)

## When To Use This Pattern

Use **Combination Sum / Backtracking with Reuse** when:

| Signal | Example |
|--------|---------|
| Build all subsets summing to target | This problem |
| Elements can be reused | Coin change variants |
| Need to enumerate, not just count | Combination problems |
| Pruning is natural (sorted input) | Any bounded numeric set |

**Contrast with:**
- LC 40 (Combination Sum II) — no reuse, skip duplicates
- LC 216 (Combination Sum III) — fixed length k
- LC 322 (Coin Change) — count only, use DP

## The Approach

**Algorithm — Backtracking with index-based deduplication:**

```
sort(candidates)
backtrack(start, path, remain):
    for i in range(start, len(candidates)):
        c = candidates[i]
        if c > remain:   # pruning: sorted, so all after are bigger
            break
        if c == remain:  # found a valid combination
            result.append(path + [c])
            break        # sorted: nothing after can help
        # c < remain: include c and recurse, reuse same index i
        path.append(c)
        backtrack(i, path, remain - c)
        path.pop()
```

**Why start from `i` (not `i+1`) on recursion?**
We allow the same candidate to be reused, so we don't advance
the index. This still avoids duplicates because we never go
*backward* in the sorted list.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    """
    Run test cases for combinationSum.
    Order-independent: each combo is sorted, then the list of
    combos is sorted before comparison.
    """
    def norm(combos):
        return sorted([sorted(c) for c in combos])

    cases = [
        # (candidates, target, expected)
        ([2, 3, 6, 7], 7,  [[2, 2, 3], [7]]),
        ([2, 3, 5],    8,  [[2, 2, 2, 2], [2, 3, 3], [3, 5]]),
        ([2],          1,  []),
        ([1],          1,  [[1]]),
        ([1],          2,  [[1, 1]]),
    ]

    passed = 0
    for i, (cands, target, expected) in enumerate(cases, 1):
        result = func(cands, target)
        ok = norm(result) == norm(expected)
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(f"  Case {i} {status}")
            print(f"    cands={cands}, target={target}")
            print(f"    expected: {norm(expected)}")
            print(f"    got:      {norm(result)}")

    total = len(cases)
    print(f"\nResult: {passed}/{total} passed",
          "✓" if passed == total else "✗")

In [6]:
def combinationSum(
    candidates: List[int], target: int
) -> List[List[int]]:
    result = []
    def backtrack(start, current, remaining):
        if remaining == 0:
            result.append(current[:])
            return
        if remaining < 0:
            return
        for i in range (start , len(candidates)):
            current.append(candidates[i])
            backtrack(i, current, remaining - candidates[i])
            current.pop()



    backtrack (0, [], target)
    return result



#Quick debug — run this cell while building
print(combinationSum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(combinationSum([2,3,5], 8))     # [[2,2,2,2],[2,3,3],[3,5]]
print(combinationSum([2], 1))          # []
print(combinationSum([1], 2))          # [[1,1]]
test_harness(combinationSum)    


[[2, 2, 3], [7]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[]
[[1, 1]]

Result: 5/5 passed ✓


In [ ]:
# Uncomment and run when solution is ready
# test_harness(combinationSum)

## Complexity

| | Value | Reason |
|-|-------|--------|
| **Time** | O(N^(T/M)) | N=candidates, T=target, M=min candidate |
| **Space** | O(T/M) | Recursion depth = target / smallest candidate |

**Pruning impact:** Sorting + early `break` dramatically reduces
the number of recursive calls explored compared to a naive DFS.
In practice, average-case performance is much better than the
worst-case bound.

## Real World Connection

**Currency denomination problems** — given coin denominations
of 1¢, 5¢, 10¢, 25¢, find all ways to make exact change for
a given amount. Reuse is allowed (you can use many pennies),
and you want all distinct multisets — exactly this pattern.

**Bill of Materials (BOM) assembly** — a manufacturer needs to
hit a precise weight or volume spec using available component
sizes, where each component can be used multiple times. The
backtracking prune-on-sort trick maps directly to cutting the
search space when a component already exceeds what is needed.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra